# test_case_results

分层展示：**case 级 -> round 级 -> task 级**（数据源：`var/runs/*/environment.json`）。


In [1]:
from pathlib import Path
import sys
import json
import subprocess
from collections import Counter

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / 'src' / 'task_router_graph').exists()
)
RUNS_DIR = PROJECT_ROOT / 'var' / 'runs'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.executable)
print('RUNS_DIR exists =', RUNS_DIR.exists())


PROJECT_ROOT = /root/WORK/task-rounting
Python = /opt/conda/envs/task-routing-clean/bin/python
RUNS_DIR exists = True


In [2]:
RUN_CASES = False
HEARTBEAT_SEC = 10
CASES_DIR = 'data/cases'

print('RUN_CASES =', RUN_CASES)
if RUN_CASES:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'run_cases.py'),
        '--cases-dir', CASES_DIR,
        '--heartbeat-sec', str(HEARTBEAT_SEC),
    ]
    print('Running:', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT)
    print('run_cases.py return code =', proc.returncode)


RUN_CASES = False


In [3]:
# 数据层：解析 + 分层索引 + 轻量展示工具（不依赖 pandas）
def _safe_dict(value):
    return value if isinstance(value, dict) else {}


def _safe_list(value):
    return value if isinstance(value, list) else []


def _short(text, limit=120):
    text = str(text).replace('\n', ' ').strip()
    if len(text) <= limit:
        return text
    return text[: limit - 3] + '...'


def render_rows(rows, columns, max_rows=50):
    print(f'rows={len(rows)} (showing up to {max_rows})')
    if not rows:
        return

    widths = {}
    for col in columns:
        max_len = len(col)
        for row in rows[:max_rows]:
            max_len = max(max_len, len(str(row.get(col, ''))))
        widths[col] = min(max_len, 48)

    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    print(header)
    print('-' * len(header))

    for row in rows[:max_rows]:
        line = []
        for col in columns:
            val = str(row.get(col, ''))
            if len(val) > widths[col]:
                val = val[: widths[col] - 3] + '...'
            line.append(val.ljust(widths[col]))
        print(' | '.join(line))


def load_case_tree(runs_dir: Path):
    case_map = {}

    for run_dir in sorted(path for path in runs_dir.glob('run_*') if path.is_dir()):
        run_token = run_dir.name
        env_path = run_dir / 'environment.json'
        if not env_path.exists():
            continue

        try:
            payload = json.loads(env_path.read_text(encoding='utf-8'))
        except Exception as exc:
            case_key = run_token
            case_item = case_map.setdefault(case_key, {'case_id': '', 'runs': []})
            case_item['runs'].append({
                'run_token': run_token,
                'updated_at': '',
                'rounds': [],
                'parse_error': f'{exc.__class__.__name__}: {exc}',
            })
            continue

        payload = _safe_dict(payload)
        case_id = str(payload.get('case_id', '')).strip()
        case_key = case_id or run_token

        run_item = {
            'run_token': run_token,
            'updated_at': payload.get('updated_at', ''),
            'rounds': [],
            'parse_error': '',
        }

        for round_item in _safe_list(payload.get('rounds')):
            round_item = _safe_dict(round_item)
            round_id = round_item.get('round_id', '')
            user_input = round_item.get('user_input', '')

            tasks = []
            for task_item in _safe_list(round_item.get('tasks')):
                task_item = _safe_dict(task_item)
                task_payload = _safe_dict(task_item.get('task'))
                trace = _safe_list(task_item.get('controller_trace'))
                last = _safe_dict(trace[-1]) if trace else {}

                tasks.append({
                    'task_id': task_item.get('task_id', task_payload.get('task_id', '')),
                    'task_type': task_payload.get('type', ''),
                    'task_status': task_payload.get('status', ''),
                    'task_content': task_payload.get('content', ''),
                    'task_result': task_payload.get('result', ''),
                    'reply': task_item.get('reply', ''),
                    'trace': trace,
                    'trace_count': len(trace),
                    'last_action_kind': last.get('action_kind', ''),
                    'last_tool': last.get('tool', ''),
                })

            run_item['rounds'].append({
                'round_id': round_id,
                'user_input': user_input,
                'tasks': tasks,
            })

        case_item = case_map.setdefault(case_key, {'case_id': case_id, 'runs': []})
        case_item['runs'].append(run_item)

    case_tree = []
    for case_key in sorted(case_map.keys()):
        case_item = case_map[case_key]
        case_tree.append({
            'case_key': case_key,
            'case_id': case_item.get('case_id', ''),
            'runs': sorted(case_item.get('runs', []), key=lambda x: x.get('run_token', '')),
        })

    return case_tree


def flatten_rounds(case_tree):
    rows = []
    for case_item in case_tree:
        case_key = case_item.get('case_key', '')
        for run_item in case_item.get('runs', []):
            run_token = run_item.get('run_token', '')
            for round_item in run_item.get('rounds', []):
                tasks = round_item.get('tasks', [])
                status_counter = Counter((str(t.get('task_status', '')).strip() or 'unknown') for t in tasks)
                rows.append({
                    'case_key': case_key,
                    'run_token': run_token,
                    'round_id': round_item.get('round_id', ''),
                    'user_input': round_item.get('user_input', ''),
                    'task_count': len(tasks),
                    'done_count': status_counter.get('done', 0),
                    'failed_count': status_counter.get('failed', 0),
                })
    return rows


def flatten_tasks(case_tree):
    rows = []
    for case_item in case_tree:
        case_key = case_item.get('case_key', '')
        for run_item in case_item.get('runs', []):
            run_token = run_item.get('run_token', '')
            for round_item in run_item.get('rounds', []):
                round_id = round_item.get('round_id', '')
                user_input = round_item.get('user_input', '')
                for task in round_item.get('tasks', []):
                    rows.append({
                        'case_key': case_key,
                        'run_token': run_token,
                        'round_id': round_id,
                        'task_id': task.get('task_id', ''),
                        'task_type': task.get('task_type', ''),
                        'task_status': task.get('task_status', ''),
                        'task_content': task.get('task_content', ''),
                        'task_result': task.get('task_result', ''),
                        'reply': task.get('reply', ''),
                        'trace_count': task.get('trace_count', 0),
                        'last_action_kind': task.get('last_action_kind', ''),
                        'last_tool': task.get('last_tool', ''),
                        'user_input': user_input,
                        'trace': task.get('trace', []),
                    })
    return rows


case_tree = load_case_tree(RUNS_DIR)
round_rows = flatten_rounds(case_tree)
task_rows = flatten_tasks(case_tree)

print('case_count =', len(case_tree))
print('round_count =', len(round_rows))
print('task_count =', len(task_rows))


case_count = 14
round_count = 14
task_count = 14


In [4]:
# CASE 级展示
case_rows = []
for case_item in case_tree:
    runs = case_item.get('runs', [])
    rounds = [rd for r in runs for rd in r.get('rounds', [])]
    tasks = [t for rd in rounds for t in rd.get('tasks', [])]

    status_counter = Counter((str(t.get('task_status', '')).strip() or 'unknown') for t in tasks)
    type_counter = Counter((str(t.get('task_type', '')).strip() or 'unknown') for t in tasks)

    case_rows.append({
        'case_key': case_item.get('case_key', ''),
        'run_count': len(runs),
        'round_count': len(rounds),
        'task_count': len(tasks),
        'done_count': status_counter.get('done', 0),
        'failed_count': status_counter.get('failed', 0),
        'task_types': str(dict(type_counter)),
    })

case_rows = sorted(case_rows, key=lambda x: str(x.get('case_key', '')))
render_rows(case_rows, ['case_key', 'run_count', 'round_count', 'task_count', 'done_count', 'failed_count', 'task_types'])


rows=14 (showing up to 50)
case_key | run_count | round_count | task_count | done_count | failed_count | task_types     
---------------------------------------------------------------------------------------------
case_01  | 1         | 1           | 1          | 1          | 0            | {'functest': 1}
case_02  | 1         | 1           | 1          | 0          | 1            | {'normal': 1}  
case_03  | 1         | 1           | 1          | 1          | 0            | {'functest': 1}
case_04  | 1         | 1           | 1          | 1          | 0            | {'functest': 1}
case_05  | 1         | 1           | 1          | 1          | 0            | {'accutest': 1}
case_06  | 1         | 1           | 1          | 1          | 0            | {'perftest': 1}
case_07  | 1         | 1           | 1          | 0          | 1            | {'normal': 1}  
case_08  | 1         | 1           | 1          | 0          | 1            | {'normal': 1}  
case_09  | 1         | 1         

In [5]:
# ROUND 级展示
CASE_FILTER = []   # 例如 ['case_01']

round_view = []
for row in round_rows:
    if CASE_FILTER and str(row.get('case_key', '')) not in CASE_FILTER:
        continue
    round_view.append({
        'case_key': row.get('case_key', ''),
        'run_token': row.get('run_token', ''),
        'round_id': row.get('round_id', ''),
        'user_input': _short(row.get('user_input', ''), 60),
        'task_count': row.get('task_count', 0),
        'done_count': row.get('done_count', 0),
        'failed_count': row.get('failed_count', 0),
    })

round_view = sorted(round_view, key=lambda x: (str(x.get('case_key', '')), str(x.get('run_token', '')), int(x.get('round_id') or 0)))
render_rows(round_view, ['case_key', 'run_token', 'round_id', 'user_input', 'task_count', 'done_count', 'failed_count'])


rows=14 (showing up to 50)
case_key | run_token           | round_id | user_input                             | task_count | done_count | failed_count
---------------------------------------------------------------------------------------------------------------------------
case_01  | run_20260408_132009 | 1        | 请帮我做一次 anthropic_ver_1 的功能测试           | 1          | 1          | 0           
case_02  | run_20260408_132016 | 1        | 请总结上一次测试结果并给出下一步建议                     | 1          | 0          | 1           
case_03  | run_20260408_132025 | 1        | 请帮我做一次 anthropic_ver_1 的功能测试           | 1          | 1          | 0           
case_04  | run_20260408_132031 | 1        | 检查一下 genai_ver_2 的 headers 和 body 行为   | 1          | 1          | 0           
case_05  | run_20260408_132038 | 1        | 对 qwen_proxy_v3 做一次准确性评估               | 1          | 1          | 0           
case_06  | run_20260408_132044 | 1        | 帮我压测 gateway_alpha，重点看 p95 和 qps       | 1          | 1      

In [6]:
# TASK 级展示
TASK_STATUS_FILTER = []   # 例如 ['failed']
TASK_TYPE_FILTER = []     # 例如 ['normal']
RUN_FILTER = []           # 例如 ['run_20260408_132016']
MAX_ROWS = 80

task_view = []
for row in task_rows:
    if CASE_FILTER and str(row.get('case_key', '')) not in CASE_FILTER:
        continue
    if TASK_STATUS_FILTER and str(row.get('task_status', '')) not in TASK_STATUS_FILTER:
        continue
    if TASK_TYPE_FILTER and str(row.get('task_type', '')) not in TASK_TYPE_FILTER:
        continue
    if RUN_FILTER and str(row.get('run_token', '')) not in RUN_FILTER:
        continue

    task_view.append({
        'case_key': row.get('case_key', ''),
        'run_token': row.get('run_token', ''),
        'round_id': row.get('round_id', ''),
        'task_id': row.get('task_id', ''),
        'task_type': row.get('task_type', ''),
        'task_status': row.get('task_status', ''),
        'task_content': _short(row.get('task_content', ''), 40),
        'task_result': _short(row.get('task_result', ''), 40),
        'trace_count': row.get('trace_count', 0),
        'last_action_kind': row.get('last_action_kind', ''),
        'last_tool': row.get('last_tool', ''),
    })

task_view = sorted(task_view, key=lambda x: (str(x.get('case_key', '')), str(x.get('run_token', '')), int(x.get('round_id') or 0), int(x.get('task_id') or 0)))
render_rows(task_view, ['case_key', 'run_token', 'round_id', 'task_id', 'task_type', 'task_status', 'task_content', 'task_result', 'trace_count', 'last_action_kind', 'last_tool'], max_rows=MAX_ROWS)


rows=14 (showing up to 80)
case_key | run_token           | round_id | task_id | task_type | task_status | task_content                             | task_result                              | trace_count | last_action_kind | last_tool
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
case_01  | run_20260408_132009 | 1        | 1       | functest  | done        | 针对 anthropic_ver_1 执行功能测试，重点验证核心功能行为与... | functest 已完成（示例执行）：针对 anthropic_ver_1... | 2           | generate_task    | None     
case_02  | run_20260408_132016 | 1        | 1       | normal    | failed      | 请总结上一次测试结果并给出下一步建议                       | route failed: ControllerAgent exceede... | 4           | observe          | None     
case_03  | run_20260408_132025 | 1        | 1       | functest  | done        | 针对 anthropic_ver_1 执行功能测试，重点验证核心功能行为与... | functest 已完成（示

In [7]:
# 树状明细：case -> round -> task -> trace（默认关闭，避免大输出）
ENABLE_TREE_VIEW = False
SHOW_TRACE = False
TRACE_LIMIT = 20

if not ENABLE_TREE_VIEW:
    print('Tree view disabled. Set ENABLE_TREE_VIEW=True to expand.')
else:
    selected_tasks = [
        t for t in task_rows
        if (not CASE_FILTER or str(t.get('case_key', '')) in CASE_FILTER)
        and (not TASK_STATUS_FILTER or str(t.get('task_status', '')) in TASK_STATUS_FILTER)
        and (not TASK_TYPE_FILTER or str(t.get('task_type', '')) in TASK_TYPE_FILTER)
        and (not RUN_FILTER or str(t.get('run_token', '')) in RUN_FILTER)
    ]

    bucket = {}
    for row in selected_tasks:
        ck = row.get('case_key', '')
        rk = row.get('run_token', '')
        rd = row.get('round_id', '')
        bucket.setdefault(ck, {}).setdefault(rk, {}).setdefault(rd, []).append(row)

    for case_key in sorted(bucket.keys()):
        print('=' * 100)
        print(f'CASE: {case_key}')
        for run_token in sorted(bucket[case_key].keys()):
            print(f'  RUN: {run_token}')
            round_keys = sorted(bucket[case_key][run_token].keys(), key=lambda x: int(x or 0))
            for round_id in round_keys:
                items = sorted(bucket[case_key][run_token][round_id], key=lambda x: int(x.get('task_id') or 0))
                user_input = items[0].get('user_input', '') if items else ''
                print(f'    ROUND {round_id}: user_input={_short(user_input, 120)}')
                for t in items:
                    print(f"      TASK#{t.get('task_id', '')} type={t.get('task_type', '')} status={t.get('task_status', '')} trace={t.get('trace_count', 0)} content={_short(t.get('task_content', ''), 90)}")
                    print(f"        result={_short(t.get('task_result', ''), 120)}")
                    if SHOW_TRACE:
                        trace = t.get('trace', [])[:TRACE_LIMIT]
                        for idx, action in enumerate(trace, start=1):
                            action = action if isinstance(action, dict) else {}
                            print(f"          [{idx}] kind={action.get('action_kind', '')} tool={action.get('tool', '')} reason={_short(action.get('reason', ''), 80)} obs={_short(action.get('observation', ''), 80)}")


Tree view disabled. Set ENABLE_TREE_VIEW=True to expand.


## 说明

- 已去掉 `pandas` 依赖，避免 `task-routing-clean` 内核因二进制包异常重启。
- 默认禁用树状大输出，先看 case/round/task 三层表；需要深挖时再开 `ENABLE_TREE_VIEW=True`。
